In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels import PanelOLS, FamaMacBeth
from tqdm.notebook import tqdm
import random
np.random.seed(809)
from scipy.stats import chi2

# Load the Data
## Factors 

In [2]:
FF3 = pd.read_csv(r'F-F_Research_Data_Factors.CSV', skiprows = 3)
FF3.rename(columns = {'Unnamed: 0':'Date'}, inplace = True)
FF3['Date'] = FF3['Date'].astype(str)
FF3.rename(columns={'Mkt-RF':'Mkt_RF'}, inplace = True)
FF3['Mkt_RF'] = FF3['Mkt_RF']/100
FF3['SMB'] = FF3['SMB']/100
FF3['HML'] = FF3['HML']/100
FF3['RF'] = FF3['RF']/100
FF3

,Date,Mkt_RF,SMB,HML,RF
0,192607,0.0296,-0.0230,-0.0287,0.0022
1,192608,0.0264,-0.0140,0.0419,0.0025
2,192609,0.0036,-0.0132,0.0001,0.0023
3,192610,-0.0324,0.0004,0.0051,0.0032
4,192611,0.0253,-0.0020,-0.0035,0.0031
...,...,...,...,...,...
1117,201908,-0.0258,-0.0241,-0.0499,0.0016
1118,201909,0.0144,-0.0090,0.0671,0.0018
1119,201910,0.0206,0.0025,-0.0207,0.0015
1120,201911,0.0387,0.0087,-0.0186,0.0012


## Stocks Return

In [3]:
Stocks = pd.read_csv('CRSP_Data.csv', low_memory=False)
Stocks['RET'] = pd.to_numeric(Stocks['RET'], errors = 'coerce')
Stocks['Date'] = Stocks['date'].astype(str).str[:6]

Stocks.drop(columns = ['date', 'PERMNO'], inplace = True)
Stocks.dropna(inplace = True)
Stocks.sort_values(['TICKER', 'Date'], inplace = True)
Stocks.drop_duplicates(subset = ['TICKER', 'Date'], inplace= True, keep =False)

In [4]:
#Make balanced: Only keep stocks for which we have each date
Stocks['count'] = Stocks.groupby('TICKER')['RET'].transform('count')
Stocks = Stocks[(Stocks['count']== Stocks['count'].max())]
Stocks.drop(columns = ['count'], inplace = True)

In [5]:
#Reduce Size: only keep 500 Tickers
nb_stocks = 500
ticker_list = Stocks['TICKER'].unique()
sublist = np.random.choice(ticker_list, nb_stocks, replace=False)
Stocks = Stocks[Stocks.TICKER.isin(sublist)]
Stocks

,TICKER,RET,Date
194007,AAPL,0.453782,200101
194008,AAPL,-0.156069,200102
194009,AAPL,0.209315,200103
194010,AAPL,0.154961,200104
194011,AAPL,-0.217340,200105
...,...,...,...
909901,ZION,-0.080763,201908
909902,ZION,0.083475,201909
909903,ZION,0.088724,201910
909904,ZION,0.034042,201911


## Merging the two

In [6]:
Stocks_FF3 = pd.merge(Stocks, FF3, on ='Date', how = 'left')
Stocks_FF3['RET_RF'] = Stocks_FF3['RET'] - Stocks_FF3['RF']
Stocks_FF3

,TICKER,RET,Date,Mkt_RF,SMB,HML,RF,RET_RF
0,AAPL,0.453782,200101,0.0313,0.0654,-0.0486,0.0054,0.448382
1,AAPL,-0.156069,200102,-0.1005,-0.0072,0.1287,0.0038,-0.159869
2,AAPL,0.209315,200103,-0.0726,0.0034,0.0646,0.0042,0.205115
3,AAPL,0.154961,200104,0.0794,0.0054,-0.0472,0.0039,0.151061
4,AAPL,-0.217340,200105,0.0072,0.0259,0.0318,0.0032,-0.220540
...,...,...,...,...,...,...,...,...
113995,ZION,-0.080763,201908,-0.0258,-0.0241,-0.0499,0.0016,-0.082363
113996,ZION,0.083475,201909,0.0144,-0.0090,0.0671,0.0018,0.081675
113997,ZION,0.088724,201910,0.0206,0.0025,-0.0207,0.0015,0.087224
113998,ZION,0.034042,201911,0.0387,0.0087,-0.0186,0.0012,0.032842


# First Regression: Time Series
## Regress each asset against the proposed risk factors to get the betas

In [7]:
#Creating dictionaries for recording the outcome from the regressions
coefs_first_stage = {}
results_first_stage = {}

for asset in tqdm(Stocks['TICKER'].unique()):    
    #One regression by asset
    df = Stocks_FF3[Stocks_FF3.TICKER ==asset]
    
    #Running the regression
    reg = smf.ols(formula='RET_RF ~ 1 + Mkt_RF + SMB + HML', data=df)
    res = reg.fit()
    
    #Record the result and the coefficients in the dictionaries
    results_first_stage[asset] = res.summary()
    coefs_first_stage[asset] = res.params[1:]

  0%|          | 0/500 [00:00<?, ?it/s]

### Output Example

In [8]:
stock_example = sublist[10]
print(f'Regression for: {stock_example}')
results_first_stage[stock_example]

Regression for: CRIS


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 RET_RF   R-squared:                       0.127
Model:                            OLS   Adj. R-squared:                  0.115
Method:                 Least Squares   F-statistic:                     10.85
Date:                Thu, 16 Feb 2023   Prob (F-statistic):           1.10e-06
Time:                        13:23:20   Log-Likelihood:                 12.893
No. Observations:                 228   AIC:                            -17.79
Df Residuals:                     224   BIC:                            -4.068
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0012      0.015     -0.075      0.941      -0.032       0.029
Mkt_RF         1.6953      0.377      4.494      0.000       0.952       2.439
SMB            1.2290      0.641      1.916      0.057      -0.035       2.493
HML           -0.4429      0.556     -0.796      0.427      -1.539       0.653
==============================================================================
Omnibus:                      108.484   Durbin-Watson:                   2.124
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              808.330
Skew:                           1.691   Prob(JB):                    2.97e-176
Kurtosis:                      11.582   Cond. No.                         43.0
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [9]:
coefs_first_stage[stock_example]

Mkt_RF    1.695341
SMB       1.228984
HML      -0.442897
dtype: float64

## Creating the average loadings on the FF3 factors for each stock

In [10]:
First_stage_df = pd.DataFrame.from_dict(coefs_first_stage, orient = 'index')
First_stage_df.columns = ['Loading_'+x for x in First_stage_df.columns]
First_stage_df.reset_index(inplace = True)
First_stage_df.rename(columns={'index':'TICKER'},inplace = True)
First_stage_df.head()

,TICKER,Loading_Mkt_RF,Loading_SMB,Loading_HML
0,AAPL,1.197580,0.326183,-0.728007
1,ABCB,0.851056,1.044445,1.292965
2,ACGL,0.419042,0.319886,0.170598
3,ACY,0.367190,0.742567,0.525983
4,ADC,0.608006,0.417878,0.564734


In [11]:
Stocks_FF3 = pd.merge(Stocks_FF3, First_stage_df, on ='TICKER', how = 'left')
Stocks_FF3

,TICKER,RET,Date,Mkt_RF,SMB,HML,RF,RET_RF,Loading_Mkt_RF,Loading_SMB,Loading_HML
0,AAPL,0.453782,200101,0.0313,0.0654,-0.0486,0.0054,0.448382,1.19758,0.326183,-0.728007
1,AAPL,-0.156069,200102,-0.1005,-0.0072,0.1287,0.0038,-0.159869,1.19758,0.326183,-0.728007
2,AAPL,0.209315,200103,-0.0726,0.0034,0.0646,0.0042,0.205115,1.19758,0.326183,-0.728007
3,AAPL,0.154961,200104,0.0794,0.0054,-0.0472,0.0039,0.151061,1.19758,0.326183,-0.728007
4,AAPL,-0.217340,200105,0.0072,0.0259,0.0318,0.0032,-0.220540,1.19758,0.326183,-0.728007
...,...,...,...,...,...,...,...,...,...,...,...
113995,ZION,-0.080763,201908,-0.0258,-0.0241,-0.0499,0.0016,-0.082363,1.04003,0.110148,1.666592
113996,ZION,0.083475,201909,0.0144,-0.0090,0.0671,0.0018,0.081675,1.04003,0.110148,1.666592
113997,ZION,0.088724,201910,0.0206,0.0025,-0.0207,0.0015,0.087224,1.04003,0.110148,1.666592
113998,ZION,0.034042,201911,0.0387,0.0087,-0.0186,0.0012,0.032842,1.04003,0.110148,1.666592


# Second Regression: Cross Section
## Regress all returns against the estimated betas

In [12]:
coefs_second_stage = {}
results_second_stage = {}
residuals = {}

for date in tqdm(Stocks['Date'].unique()):
    #One regression per date
    df = Stocks_FF3[Stocks_FF3['Date']==date]
    
    #Running the regression
    #Note that since we are working with excess returns, we do not include an intercept (hence the -1)
    reg = smf.ols(formula='RET_RF ~ Loading_Mkt_RF + Loading_SMB + Loading_HML -1', data=df)
    res = reg.fit()
    
    #Record the result and the coefficients
    results_second_stage[date] = res.summary()
    coefs_second_stage[date] = res.params
    residuals[date] = res.resid.reset_index(drop=True)

  0%|          | 0/228 [00:00<?, ?it/s]

### Output Example

In [13]:
date_example = list(Stocks['Date'].unique())[0]
print(f'Regression for: {date_example}')
results_second_stage[date_example]

Regression for: 200101


<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                 RET_RF   R-squared (uncentered):                   0.392
Model:                            OLS   Adj. R-squared (uncentered):              0.388
Method:                 Least Squares   F-statistic:                              106.6
Date:                Thu, 16 Feb 2023   Prob (F-statistic):                    2.72e-53
Time:                        13:23:26   Log-Likelihood:                          23.680
No. Observations:                 500   AIC:                                     -41.36
Df Residuals:                     497   BIC:                                     -28.72
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Loading_Mkt_RF     0.0300      0.015      1.961      0.050    -6.4e-05       0.060
Loading_SMB        0.2248      0.020     11.398      0.000       0.186       0.264
Loading_HML       -0.1419      0.018     -7.930      0.000      -0.177      -0.107
==============================================================================
Omnibus:                      381.687   Durbin-Watson:                   2.032
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            10414.065
Skew:                           3.012   Prob(JB):                         0.00
Kurtosis:                      24.531   Cond. No.                         2.77
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Average Values and Variances

In [14]:
Second_stage_coefs = pd.DataFrame.from_dict(coefs_second_stage, orient = 'index')
Second_stage_coefs.head()

,Loading_Mkt_RF,Loading_SMB,Loading_HML
200101,0.029952,0.224850,-0.141870
200102,-0.082291,-0.022358,0.143175
200103,-0.068517,0.013860,0.063901
200104,0.118182,-0.005261,-0.080904
200105,0.002563,0.073972,0.013864


### Means, Variances and T-Stats
Note that we have an $T$ in the denominator of the variances, because it is the variance of the mean rather than the variance of the estimates themselves that we are interested in.

In [15]:
Second_Stage_Mean_Variance = pd.DataFrame()
Second_Stage_Mean_Variance['Mean'] = Second_stage_coefs.mean()
Second_Stage_Mean_Variance['Variance'] = Second_stage_coefs.var()/[len(Stocks['Date'].unique())]
Second_Stage_Mean_Variance['T-Stat'] = Second_Stage_Mean_Variance['Mean']/np.sqrt(Second_Stage_Mean_Variance['Variance'])
Second_Stage_Mean_Variance

,Mean,Variance,T-Stat
Loading_Mkt_RF,0.007203,0.000009,2.434699
Loading_SMB,0.005254,0.000004,2.594979
Loading_HML,-0.000240,0.000006,-0.101256


# Pricing Errors
For each stock, we can compute the average pricing error

In [16]:
residuals_df = pd.DataFrame.from_dict(residuals, orient = 'index')
residuals_df.columns= list(df.TICKER.unique())
residuals_df

,AAPL,ABCB,ACGL,ACY,ADC,ADM,ADP,ADTN,AEIS,AEP,...,WSTL,WTFC,WY,XLE,XLI,XLP,XLV,XRX,YPF,ZION
200101,0.235887,0.064365,-0.036508,0.020400,0.118183,0.044617,-0.053372,0.090084,-0.029330,-0.005064,...,-0.000019,0.192925,0.046665,-0.066846,-0.009628,0.012740,0.054462,0.569008,0.060419,0.070016
200102,0.050206,-0.037394,0.049847,-0.060629,-0.018356,0.054958,0.067013,-0.008176,-0.043631,0.119241,...,0.178812,-0.145963,0.049533,0.026988,0.001119,0.031050,0.054922,-0.056333,0.016251,-0.122822
200103,0.329169,-0.021447,0.001364,-0.002111,0.105580,-0.078385,-0.009962,0.115401,0.261685,0.010651,...,-0.214447,-0.037292,-0.008875,0.010412,-0.035604,-0.031316,-0.006646,0.122366,-0.014383,-0.134064
200104,-0.047654,-0.020465,-0.039513,-0.021242,-0.077000,-0.181429,-0.120104,-0.048602,0.101883,0.000987,...,-0.714066,0.103504,0.004436,0.012231,-0.006069,-0.051281,-0.029910,0.276329,-0.046642,0.031639
200105,-0.237644,-0.075565,0.026481,-0.024695,0.075651,0.157259,0.004129,-0.124606,-0.160681,0.043107,...,0.344395,0.210425,-0.004133,-0.031129,0.032666,0.041389,0.044530,0.064842,0.024554,-0.017415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201908,0.027493,-0.053265,0.042659,-0.178693,0.154307,-0.036504,0.055482,-0.022053,-0.026679,0.068051,...,-0.189458,-0.062958,0.101310,-0.037367,0.026977,0.047506,0.024758,-0.023654,-0.421352,-0.007446
201909,0.070044,0.055526,0.039439,-0.066309,-0.060582,0.061545,-0.062613,0.076102,0.059368,0.006886,...,-0.042922,-0.058684,0.003392,-0.000529,-0.011372,-0.001486,-0.016308,0.006599,0.020727,-0.025625
201910,0.072890,0.066884,-0.012729,-0.039042,0.074020,0.016206,-0.008702,-0.255293,-0.019643,0.006035,...,-0.065796,-0.004915,0.045449,-0.031824,-0.001758,-0.007275,0.037956,0.096449,0.008877,0.107267
201911,0.036253,-0.005202,-0.008770,-0.021219,-0.069639,0.005014,0.022337,0.013803,0.025099,-0.042198,...,-0.248490,0.047285,-0.030952,-0.014471,0.007417,-0.008109,0.023636,0.090549,-0.009078,0.002155


In [17]:
# Average pricing error by asset
Pricing_Errors = pd.DataFrame()
Pricing_Errors['Mean Pricing Error'] = residuals_df.mean(axis =0)
Pricing_Errors['Variance Pricing Error'] = (residuals_df.var()/len(Stocks['Date'].unique()))
Pricing_Errors['T-Stat'] = Pricing_Errors['Mean Pricing Error']/np.sqrt(Pricing_Errors['Variance Pricing Error'])
Pricing_Errors

,Mean Pricing Error,Variance Pricing Error,T-Stat
AAPL,0.019325,0.000033,3.345352
ABCB,0.001532,0.000023,0.316869
ACGL,0.009816,0.000010,3.094106
ACY,0.004581,0.000109,0.438290
ADC,0.008432,0.000023,1.757848
...,...,...,...
XLP,0.002855,0.000002,1.844098
XLV,0.001986,0.000003,1.069985
XRX,-0.003155,0.000033,-0.552347
YPF,0.000971,0.000061,0.124237


# Second Stage Using Linearmodels

In [18]:
Stocks_FF3_FE = Stocks_FF3.copy()
Stocks_FF3_FE['Date'] = Stocks_FF3_FE['Date'].astype(int)
Stocks_FF3_FE.set_index(['TICKER','Date'], inplace=True)

mod = FamaMacBeth.from_formula('RET_RF ~ Loading_Mkt_RF + Loading_SMB + Loading_HML - 1', Stocks_FF3_FE)
res = mod.fit()
res

C:\Users\aju17\AppData\Local\Continuum\anaconda3\lib\site-packages\linearmodels\panel\data.py:98: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  if is_categorical(s):


Dep. Variable:,RET_RF,R-squared:,0.0078
Estimator:,FamaMacBeth,R-squared (Between):,0.7215
No. Observations:,114000,R-squared (Within):,1.11e-16
Date:,"Thu, Feb 16 2023",R-squared (Overall):,0.0078
Time:,13:23:31,Log-likelihood,7.977e+04
Cov. Estimator:,Fama-MacBeth Standard Cov,,
,,F-statistic:,299.84
Entities:,500,P-value,0.0000
Avg Obs:,228.00,Distribution:,"F(3,113997)"
Min Obs:,228.00,,
Max Obs:,228.00,F-statistic (robust):,3.5446


# Full Fama MacBeth Using Linear Models (first and second stage at once)
Note that the T-Stats are slightly different because of the robust standard errors

In [19]:
#Setting the data in the proper way
Portfolios = Stocks_FF3_FE['RET_RF'].unstack().T
temp = Stocks_FF3_FE.reset_index()
Factors = temp[temp['TICKER']==temp['TICKER'].iloc[0]][['Date', 'HML', 'SMB', 'Mkt_RF']].set_index('Date')

In [20]:
Portfolios

TICKER,AAPL,ABCB,ACGL,ACY,ADC,ADM,ADP,ADTN,AEIS,AEP,...,WSTL,WTFC,WY,XLE,XLI,XLP,XLV,XRX,YPF,ZION
Date,,,,,,,,,,,,,,,,,,,,,
200101,0.448382,0.141267,0.023767,0.123743,0.150236,-0.006733,-0.059931,0.318129,0.358489,-0.075292,...,0.484396,0.194600,0.036965,-0.037264,-0.001560,-0.078310,0.066894,0.761087,-0.003312,-0.110505
200102,-0.159869,0.054340,0.032637,-0.032140,0.003123,0.004238,-0.018167,-0.157133,-0.264492,0.109264,...,0.013323,-0.036088,0.019819,-0.015938,-0.063730,0.002242,-0.022965,-0.258390,0.015200,0.027744
200103,0.205115,0.017338,-0.012013,0.016633,0.105800,-0.130446,-0.080768,0.027296,0.133541,-0.015559,...,-0.276927,0.005969,-0.059094,-0.033923,-0.099853,-0.073552,-0.068223,-0.012478,-0.050328,-0.097301
200104,0.151061,-0.029987,-0.005475,-0.024308,-0.053033,-0.098197,-0.006291,0.109586,0.340797,0.045887,...,-0.580196,0.080664,0.109114,0.099357,0.110429,0.015518,0.069925,0.505282,0.032369,0.019137
200105,-0.220540,0.021800,0.053582,0.038467,0.115949,0.134499,-0.012601,-0.072670,-0.061108,0.026391,...,0.479317,0.284424,0.015905,-0.012648,0.038650,0.011482,0.034994,0.093039,0.032514,0.016503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201908,-0.018170,-0.116762,0.019336,-0.206600,0.115678,-0.066839,0.018337,-0.077207,-0.117353,0.044067,...,-0.239169,-0.119856,0.033819,-0.084854,-0.028071,0.020106,-0.007525,-0.098485,-0.481235,-0.082363
201909,0.071162,0.145969,0.060985,-0.026957,-0.014787,0.077569,-0.046725,0.102874,0.109935,0.026066,...,-0.016293,0.026849,0.063955,0.037874,0.028355,0.015621,-0.002799,0.038559,0.078807,0.081675
201910,0.109184,0.063361,-0.006741,-0.036403,0.075328,0.022118,0.003518,-0.224946,0.027937,0.005971,...,-0.023559,-0.014033,0.053013,-0.022446,0.009836,-0.005733,0.049759,0.132903,0.010392,0.087224


In [21]:
Factors

,HML,SMB,Mkt_RF
Date,,,
200101,-0.0486,0.0654,0.0313
200102,0.1287,-0.0072,-0.1005
200103,0.0646,0.0034,-0.0726
200104,-0.0472,0.0054,0.0794
200105,0.0318,0.0259,0.0072
...,...,...,...
201908,-0.0499,-0.0241,-0.0258
201909,0.0671,-0.0090,0.0144
201910,-0.0207,0.0025,0.0206


In [22]:
from linearmodels.asset_pricing.model import LinearFactorModel, LinearFactorModelGMM
mod = LinearFactorModel(Portfolios, Factors, risk_free=False)
res = mod.fit()
res

'''
Note: risk_free is Flag indicating whether the risk-free rate should be 
estimated from returns along other risk premia. 
If False, the returns are assumed to be excess returns using the correct risk-free rate
'''
print(res)

C:\Users\aju17\AppData\Local\Continuum\anaconda3\lib\site-packages\linearmodels\iv\data.py:25: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  if is_categorical(s):


                      LinearFactorModel Estimation Summary                      
No. Test Portfolios:                500   R-squared:                      0.2080
No. Factors:                          3   J-statistic:                    932.19
No. Observations:                   228   P-value                         0.0000
Date:                  Thu, Feb 16 2023   Distribution:                chi2(497)
Time:                          13:23:32                                         
Cov. Estimator:                  robust                                         
                                                                                
                            Risk Premia Estimates                             
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
HML           -0.0002     0.0025    -0.0971     0.9226     -0.0051      0.0046
SMB            0.0053     0.0022    

In [25]:
res.full_summary

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                      LinearFactorModel Estimation Summary                      
================================================================================
No. Test Portfolios:                500   R-squared:                      0.2080
No. Factors:                          3   J-statistic:                    932.19
No. Observations:                   228   P-value                         0.0000
Date:                  Thu, Feb 16 2023   Distribution:                chi2(497)
Time:                          13:23:32                                         
Cov. Estimator:                  robust                                         
                                                                                
                            Risk Premia Estimates                             
==============================================================================
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
HML           -0.0002     0.0025    -0.0971     0.9226     -0.0051      0.0046
SMB            0.0053     0.0022     2.3559     0.0185      0.0009      0.0096
Mkt_RF         0.0072     0.0029     2.4874     0.0129      0.0015      0.0129


                              AAPL Coefficients                               
==============================================================================
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
alpha          0.0193     0.0060     3.1955     0.0014      0.0075      0.0312
HML           -0.7280     0.2407    -3.0249     1.9975     -1.1997     -0.2563
SMB            0.3262     0.2718     1.2002     0.2300     -0.2065      0.8588
Mkt_RF         1.1976     0.1598     7.4959     0.0000      0.8844      1.5107


                              ABCB Coefficients                               
==============================================================================
alpha          0.0015     0.0054     0.2836     0.7767     -0.0091      0.0121
HML            1.2930     0.2291     5.6441     0.0000      0.8440      1.7420
SMB            1.0444     0.2155     4.8470     0.0000      0.6221      1.4668
Mkt_RF         0.8511     0.1610     5.2860     0.0000      0.5355      1.1666


                              ACGL Coefficients                               
==============================================================================
alpha          0.0098     0.0028     3.4451     0.0006      0.0042      0.0154
HML            0.1706     0.2117     0.8059     0.4203     -0.2443      0.5855
SMB            0.3199     0.2412     1.3264     0.1847     -0.1528      0.7926
Mkt_RF         0.4190     0.0706     5.9317     0.0000      0.2806      0.5575


                               ACY Coefficients                               
==============================================================================
alpha          0.0046     0.0105     0.4366     0.6624     -0.0160      0.0251
HML            0.5260     0.2730     1.9267     0.0540     -0.0091      1.0610
SMB            0.7426     0.3966     1.8721     0.0612     -0.0348      1.5200
Mkt_RF         0.3672     0.2682     1.3691     0.1710     -0.1585      0.8928


                               ADC Coefficients                               
==============================================================================
alpha          0.0084     0.0050     1.6963     0.0898     -0.0013      0.0182
HML            0.5647     0.2347     2.4058     0.0161      0.1046      1.0248
SMB            0.4179     0.2041     2.0473     0.0406      0.0178      0.8179
Mkt_RF         0.6080     0.1507     4.0339     0.0001      0.3126      0.9034


                               ADM Coefficients                               
=============================================